In [1]:
# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

Cloning into '/content/Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 206 (delta 104), reused 81 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 7.53 MiB | 19.27 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [2]:
# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00


In [3]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

import importlib
import captioning
importlib.reload(captioning)
from captioning import run_captioning

In [ ]:
# =============================
# CONTROLLED VERBOSITY
# =============================

# Disable HF download progress bars BEFORE importing anything HF-related

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [4]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import random
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging


# Silence transformers & HF logs (keep only errors)
transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [5]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

'/content/Project_Generative_AI_for_Data_Augmentation'

In [ ]:
# control flags
RUN_CAPTIONING = True
RUN_TEXT_VARIATION = False
RUN_IMAGE_GENERATION = False
RUN_TRAINING = False

In [6]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

100%|██████████| 792M/792M [00:44<00:00, 17.6MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 10.3MB/s]


Train size: 3680
Test size: 3669


In [7]:
# extract labels
labels = dataset_train._labels
indices = np.arange(len(dataset_train))

# perform stratified split
train_small_idx, _ = train_test_split(
    indices,
    train_size=0.30,
    stratify=labels,
    random_state=42
)

SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

np.save(os.path.join(SPLIT_DIR, "train_small_indices.npy"), train_small_idx)

In [8]:
# # run for ALL
# # create subset dataset from training set
dataset_train_small = Subset(dataset_train, train_small_idx)

In [9]:
# run for 10
dataset_train_small_10 = Subset(dataset_train, train_small_idx[:10])

# Captioning

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16 # to reduce GPU memory usage
)

model.to(device)
model.eval()

Blip2ForConditionalGeneration(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (qkv): Linear(in_features=1408, out_features=4224, bias=True)
            (projection): Linear(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
  )
  (qf

In [ ]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

if RUN_CAPTIONING:
    captions_dict = run_captioning(
        # dataset_train_small=dataset_train_small, # FOR ALL
        dataset_train_small=dataset_train_small_10, # FOR 10
        model=model,
        processor=processor,
        device=device,
        output_path=CAPTION_PATH
    )

100%|██████████| 10/10 [00:08<00:00,  1.13it/s]

Full caption generation completed.


In [14]:
import nbformat
import os

def hard_clean_notebook(path):
    nb = nbformat.read(path, as_version=4)

    if "widgets" in nb.metadata:
        del nb.metadata["widgets"]

    for cell in nb.cells:
        if "widgets" in cell.get("metadata", {}):
            del cell["metadata"]["widgets"]

    nbformat.write(nb, path)
    print(f"Cleaned: {os.path.basename(path)}")


# 🔥 Walk entire project and clean every notebook
for root, _, files in os.walk(PROJECT_ROOT):
    for file in files:
        if file.endswith(".ipynb"):
            hard_clean_notebook(os.path.join(root, file))

print("All notebooks in project HARD cleaned.")

Cleaned: main.ipynb
Cleaned: 01_captioning.ipynb
Cleaned: 03_image_generation.ipynb
Cleaned: 04_training_evaluation.ipynb
Cleaned: 02_text_variation.ipynb
All notebooks in project HARD cleaned.


In [ ]:
! git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   src/captioning.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/captions/captions_train_small_10.json

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
! git add .

In [ ]:
! git config --global user.email "fellinegiorgia@gmail.com"
! git config --global user.name "Jorj91"


In [ ]:
! git commit -m "Update captioning pipeline and add 10-sample caption output"

[main bc6b90a] Update captioning pipeline and add 10-sample caption output
 2 files changed, 222 insertions(+), 3 deletions(-)
 create mode 100644 data/captions/captions_train_small_10.json
 rewrite src/captioning.py (95%)


In [ ]:
!git pull origin main --rebase

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 4.58 KiB | 4.58 MiB/s, done.
From https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation
 * branch            main       -> FETCH_HEAD
   46cd330..c5dfb0c  main       -> origin/main
Successfully rebased and updated refs/heads/main.


In [ ]:
!git push https://MYTOKEN@github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git main

Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 12 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 2.41 KiB | 2.41 MiB/s, done.
Total 7 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git
   c5dfb0c..eecc582  main -> main


# Text variation

## FLAN-T5-Large Model



In [10]:
import text_variation_flan_large
importlib.reload(text_variation_flan_large)
from text_variation_flan_large import run_text_variation as run_flan_large

In [11]:
MAX_ITEMS = 10

In [12]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [13]:
print("\n=== FLAN LARGE ===")
run_flan_large(
    caption_file=CAPTION_PATH,
    max_items=MAX_ITEMS
)


=== FLAN LARGE ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


Original: a saint bernard dog is standing in the snow
Generated: ['christmas holiday greeting on christmas night without rain falling when an american pet stand above . by catineireren_pics-brytchescanalsv iq-', 'in autumn with little kids and cats dog and children by mountains full sunny sun ; it standing very wide green fur for', 'The young adult standing and kisse the pout with people who also live outdoors outdoors who hold her as someone smile by standing while wearing thick boots next is one male of all time from east London']

Original: a havanese dog is standing on a wooden staircase
Generated: ['We find two beautiful looking and cub obsessed Hashari pupa on stone at an arch at man made paw ramp garden area indoor and outdoor playground to go hiking in autumn and the summer in old', 'Two dogs have started walking out front under glass roof top covered raileels over white roof posts like in previous day and now with wire mesh across two pole posts, standing side as seen the whi

 90%|█████████ | 9/10 [00:21<00:02,  2.17s/it]


Original: a havanese dog is sitting on the steps
Generated: ['He had no luck on walks as there appears this girl next up this. the black is dog and looks cute outside by that park and looks out there looking as taller that normal it will have never', 'hay hastled and dogs lay there without rest while they were all outside near giant tall, tower like gates that stood for half. A black haiana sitting upright as humans have done some very', 'there will at very end with your dog watching in']

Original: a persian cat is looking at the camera with an angry expression
Generated: ['the red leopard on grass wearing grey glasses catches our very happy lisam from above and smile the cats owner to no response that you don * # love kitten like you did her but now', 'black animal in motion showing you the different patterns that lie deep underwater during photo sessions near its location in open salt sea below us country capital country', 'Several large kitten sitting beside two kitten pangler at 

100%|██████████| 10/10 [00:24<00:00,  2.41s/it]


Original: a persian cat is looking angry
Generated: ['portrait portrait of sad petper and an american flag to his pet black c. from photo album and to cat from black camera album and photograph. caption photo portrait with felin imagen &', 'male per capitai wearing black shirt runs on dirty ground by grass is looking at an inconsider', 'looking sicky little pericy cat that says that all is done to keep your money nice for today,']
